# Assignment 6 — Grapheme-to-Phoneme (G2P) using Phrase-Based SMT

**Yee Mon Thant**

**Course:** AI Engineering Foundations (AIE-F)

**Taught by:** Sayar Ye Kyaw Thu, Language Understanding Lab (LU Lab.), Myanmar

**Data:** myG2P corpus — https://github.com/ye-kyaw-thu/myG2P

## Overview

This notebook implements Grapheme-to-Phoneme (G2P) conversion using the Moses
Phrase-Based Statistical Machine Translation (PBSMT) framework.

G2P conversion predicts how a written word is pronounced. It is a core component
of ASR and TTS systems. Here it is framed as a translation problem:
- **Source:** Myanmar grapheme syllables (written form, `.my` files)
- **Target:** Romanized phoneme transcriptions (pronunciation, `.ph` files)



**Two translation directions:**
1. `my → ph` — grapheme to phoneme
2. `ph → my` — phoneme to grapheme (reverse)



## Step 1 — Environment Check

In [1]:
!uname -a
!python --version

Linux 45d707d429c9 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
Python 3.12.13


## Step 2 — Install GIZA++ and Download Moses

- Clones and compiles GIZA++ from source (IBM Model 1–4 word aligner)
- Downloads the pre-built Moses Ubuntu 17.04 binary
- Copies all required tools into one directory



In [2]:
%%bash
set -e

echo "=== Cleaning up any previous files ==="
rm -rf /content/giza-pp
rm -rf /content/ubuntu-17.04
rm -f /content/ubuntu-17.04.tgz

echo "=== Step 2a: Compiling GIZA++ ==="
git clone https://github.com/moses-smt/giza-pp.git
cd giza-pp && make -s
echo "GIZA++ compiled"

echo "=== Step 2b: Downloading Moses binary ==="
cd /content
wget -q --show-progress https://www.statmt.org/moses-release/RELEASE-4.0/binaries/ubuntu-17.04.tgz
tar -xzf ubuntu-17.04.tgz
echo "Moses downloaded"

echo "=== Step 2c: Copying tools to one directory ==="
cp /content/giza-pp/GIZA++-v2/snt2cooc.out /content/giza-pp/GIZA++-v2/snt2cooc
cp /content/giza-pp/mkcls-v2/mkcls         /content/giza-pp/GIZA++-v2/

echo ""
echo "=== ALL SETUP DONE ==="

=== Cleaning up any previous files ===
=== Step 2a: Compiling GIZA++ ===
GIZA++ compiled
=== Step 2b: Downloading Moses binary ===
Moses downloaded
=== Step 2c: Copying tools to one directory ===

=== ALL SETUP DONE ===


Cloning into 'giza-pp'...
Parameter.cpp: In function ‘bool writeParameters(std::ofstream&, const ParSet&, int)’:
Parameter.cpp:48:21: warning: ignoring return value of ‘char* getcwd(char*, size_t)’ declared with attribute ‘warn_unused_result’ [-Wunused-result]
   48 |               getcwd(path,1024);
      |               ~~~~~~^~~~~~~~~~~

     0K .......... .......... .......... .......... ..........  0%  112K 17m28s
    50K .......... .......... .......... .......... ..........  0% 1.24M 9m30s
   100K .......... .......... .......... .......... ..........  0%  272K 8m44s
   150K .......... .......... .......... .......... ..........  0%  125M 6m33s
   200K .......... .......... .......... .......... ..........  0%  224K 6m59s
   250K .......... .......... .......... .......... ..........  0% 80.2M 5m49s
   300K .......... .......... .......... .......... ..........  0%  104M 4m59s
   350K .......... .......... .......... .......... ..........  0%  189M 4m22s
   400K .......... .....

## Step 3 — Upload G2P Parallel Corpus

Upload all 6 files: `train.my`, `train.ph`, `dev.my`, `dev.ph`, `test.my`, `test.ph`





In [3]:
from google.colab import files
import os, shutil

os.makedirs('/content/g2p-par', exist_ok=True)
uploaded = files.upload()

for f in uploaded.keys():
    shutil.move(f, f'/content/g2p-par/{f}')
    print(f"Saved: {f}")

print("\nFiles in /content/g2p-par/:")
print(sorted(os.listdir('/content/g2p-par')))

Saving dev.my to dev.my
Saving dev.ph to dev.ph
Saving test.my to test.my
Saving test.ph to test.ph
Saving train.my to train.my
Saving train.ph to train.ph
Saved: dev.my
Saved: dev.ph
Saved: test.my
Saved: test.ph
Saved: train.my
Saved: train.ph

Files in /content/g2p-par/:
['dev.my', 'dev.ph', 'test.my', 'test.ph', 'train.my', 'train.ph']


## Step 4 — Train Language Model (my→ph direction)

A trigram language model is trained on the phoneme side (`train.ph`).
The LM scores how natural a phoneme sequence sounds, helping the decoder
choose fluent output when multiple candidates score similarly.

- `lmplz -o 3` trains a 3-gram KenLM model (ARPA format)
- `build_binary` converts it to fast binary format for decoding



In [4]:
%%bash
mkdir -p /content/model/my-ph/lm

echo "Training 3-gram language model on phoneme side..."
/content/ubuntu-17.04/moses/bin/lmplz \
  -o 3 \
  < /content/g2p-par/train.ph \
  > /content/model/my-ph/lm/ph.arpa 2>/dev/null

echo "Converting to binary format..."
/content/ubuntu-17.04/moses/bin/build_binary \
  /content/model/my-ph/lm/ph.arpa \
  /content/model/my-ph/lm/ph.blm 2>/dev/null

echo "Binary LM ready"
ls /content/model/my-ph/lm/

Training 3-gram language model on phoneme side...
Converting to binary format...
Binary LM ready
ph.arpa
ph.blm


## Step 5 — Moses Training: my→ph Baseline

`train-model.perl` runs the full PBSMT pipeline in 9 steps:

| Step | Description |
|---|---|
| 1 | Prepare corpus and vocabulary files |
| 2 | Run GIZA++ — word alignment in both directions |
| 3 | Symmetrize alignments (grow-diag-final heuristic) |
| 4 | Extract lexical translation table |
| 5 | Extract phrase pairs consistent with alignment |
| 6 | Score phrase pairs (Good-Turing smoothing) |
| 7 | Reordering model — skipped (G2P is monotone) |
| 8 | Generation model — skipped |
| 9 | Write `moses.ini` decoder config file |




In [5]:
%%bash
mkdir -p /content/model/my-ph

echo "Starting Moses training: my -> ph"
perl /content/ubuntu-17.04/moses/scripts/training/train-model.perl \
  --root-dir /content/model/my-ph \
  --corpus /content/g2p-par/train \
  --f my \
  --e ph \
  --mgiza \
  --mgiza-cpus 2 \
  --external-bin-dir /content/ubuntu-17.04/training-tools \
  --score-options '--GoodTuring' \
  --lm 0:3:/content/model/my-ph/lm/ph.blm \
  2>&1 | grep -E "^\([0-9]\)|ERROR"

echo "Training complete"

Starting Moses training: my -> ph
(1) preparing corpus @ Wed Aug  5 07:40:10 UTC 2026
(2) running giza @ Wed Aug  5 07:40:21 UTC 2026
(3) generate word alignment @ Wed Aug  5 07:40:27 UTC 2026
(4) generate lexical translation table 0-0 @ Wed Aug  5 07:40:28 UTC 2026
(5) extract phrases @ Wed Aug  5 07:40:28 UTC 2026
(6) score phrases @ Wed Aug  5 07:40:29 UTC 2026
(7) learn reordering model @ Wed Aug  5 07:40:32 UTC 2026
(8) learn generation model @ Wed Aug  5 07:40:32 UTC 2026
(9) create moses.ini @ Wed Aug  5 07:40:32 UTC 2026
Training complete


## Step 6 — Check Moses Config File

In [6]:
%%bash
echo "=== Model files ==="
ls /content/model/my-ph/model/

echo ""
echo "=== moses.ini ==="
cat /content/model/my-ph/model/moses.ini

=== Model files ===
aligned.grow-diag-final
extract.inv.sorted.gz
extract.sorted.gz
lex.e2f
lex.f2e
moses.ini
phrase-table.gz

=== moses.ini ===
#########################
### MOSES CONFIG FILE ###
#########################

# input factors
[input-factors]
0

# mapping steps
[mapping]
0 T 0

[distortion-limit]
6

# feature functions
[feature]
UnknownWordPenalty
WordPenalty
PhrasePenalty
PhraseDictionaryMemory name=TranslationModel0 num-features=4 path=/content/model/my-ph/model/phrase-table.gz input-factor=0 output-factor=0
Distortion
KENLM name=LM0 factor=0 path=/content/model/my-ph/lm/ph.blm order=3

# dense weights for feature functions
[weight]
# The default weights are NOT optimized for translation quality. You MUST tune the weights.
# Documentation for tuning is here: http://www.statmt.org/moses/?n=FactoredTraining.Tuning 
UnknownWordPenalty0= 1
WordPenalty0= -1
PhrasePenalty0= 0.2
TranslationModel0= 0.2 0.2 0.2 0.2
Distortion0= 0.3
LM0= 0.5


## Step 7 — Decode Test Set (my→ph Baseline)

The decoder translates each line of `test.my` into phonemes using
the trained phrase table and language model.


In [7]:
%%bash
echo "Decoding test set (my -> ph)..."
/content/ubuntu-17.04/moses/bin/moses \
  -f /content/model/my-ph/model/moses.ini \
  < /content/g2p-par/test.my \
  > /content/model/my-ph/model/test.out \
  2>/content/model/my-ph/model/decode.log

echo ""
echo "=== First 5 decoded outputs ==="
head -5 /content/model/my-ph/model/test.out

echo ""
echo "=== First 5 gold references ==="
head -5 /content/g2p-par/test.ph

Decoding test set (my -> ph)...

=== First 5 decoded outputs ===
te' te' pjaun 
ka' pi. 
shoun. me. 
njhin: ban: 
mun: man 

=== First 5 gold references ===
te' te' pjaun
ka' pi.
shoun. me.
njhin: ban:
mun: man


## Step 8 — BLEU Evaluation: my→ph Baseline

BLEU measures n-gram overlap (unigram through 4-gram) between the decoded
output and the gold reference, with a brevity penalty. Higher is better.


In [8]:
%%bash
echo "=== BLEU score: my -> ph (baseline) ==="
perl /content/ubuntu-17.04/moses/scripts/generic/multi-bleu.perl \
  /content/g2p-par/test.ph \
  < /content/model/my-ph/model/test.out

=== BLEU score: my -> ph (baseline) ===
BLEU = 63.35, 84.2/66.1/56.7/51.0 (BP=1.000, ratio=1.000, hyp_len=8050, ref_len=8048)


## Step 9 — MERT Tuning: my→ph Improved



MERT (Minimum Error Rate Training) finds better weights by running the decoder
repeatedly on the dev set and maximizing BLEU. This typically improves BLEU by 2–5 points.



In [9]:
%%bash
echo "Starting MERT tuning on dev set..."
perl /content/ubuntu-17.04/moses/scripts/training/mert-moses.pl \
  /content/g2p-par/dev.my \
  /content/g2p-par/dev.ph \
  /content/ubuntu-17.04/moses/bin/moses \
  /content/model/my-ph/model/moses.ini \
  --mertdir /content/ubuntu-17.04/moses/bin/ \
  --working-dir /content/model/my-ph/mert \
  2>&1 | grep -E "Best|BLEU|Iter|error|round"

echo "MERT finished"

Starting MERT tuning on dev set...
exec: /content/ubuntu-17.04/moses/bin/mert -d 8  --sctype BLEU --scconfig case:true --ffile run1.features.dat --scfile run1.scores.dat --ifile run1.init.opt -n 20
Executing: /content/ubuntu-17.04/moses/bin/mert -d 8  --sctype BLEU --scconfig case:true --ffile run1.features.dat --scfile run1.scores.dat --ifile run1.init.opt -n 20 > mert.out 2> mert.log
exec: /content/ubuntu-17.04/moses/bin/mert -d 8  --sctype BLEU --scconfig case:true --ffile run1.features.dat,run2.features.dat --scfile run1.scores.dat,run2.scores.dat --ifile run2.init.opt -n 20
Executing: /content/ubuntu-17.04/moses/bin/mert -d 8  --sctype BLEU --scconfig case:true --ffile run1.features.dat,run2.features.dat --scfile run1.scores.dat,run2.scores.dat --ifile run2.init.opt -n 20 > mert.out 2> mert.log
exec: /content/ubuntu-17.04/moses/bin/mert -d 8  --sctype BLEU --scconfig case:true --ffile run1.features.dat,run2.features.dat,run3.features.dat --scfile run1.scores.dat,run2.scores.dat,ru

## Step 10 — Decode and Evaluate with Tuned Weights (my→ph)

In [10]:
%%bash
echo "Decoding with tuned weights..."
/content/ubuntu-17.04/moses/bin/moses \
  -f /content/model/my-ph/mert/moses.ini \
  < /content/g2p-par/test.my \
  > /content/model/my-ph/model/test.tuned.out \
  2>/dev/null

echo "=== BLEU score: my -> ph (MERT tuned) ==="
perl /content/ubuntu-17.04/moses/scripts/generic/multi-bleu.perl \
  /content/g2p-par/test.ph \
  < /content/model/my-ph/model/test.tuned.out

Decoding with tuned weights...
=== BLEU score: my -> ph (MERT tuned) ===
BLEU = 69.88, 85.3/72.8/65.0/59.2 (BP=1.000, ratio=1.000, hyp_len=8050, ref_len=8048)


## Step 11 — Train Language Model (ph→my direction)

Same pipeline repeated for the reverse direction: phoneme → grapheme.
The LM is now trained on the grapheme side (`train.my`).




In [11]:
%%bash
mkdir -p /content/model/ph-my/lm

echo "Training 3-gram language model on grapheme side..."
/content/ubuntu-17.04/moses/bin/lmplz \
  -o 3 \
  < /content/g2p-par/train.my \
  > /content/model/ph-my/lm/my.arpa 2>/dev/null

/content/ubuntu-17.04/moses/bin/build_binary \
  /content/model/ph-my/lm/my.arpa \
  /content/model/ph-my/lm/my.blm 2>/dev/null

echo "Binary LM ready"
ls /content/model/ph-my/lm/

Training 3-gram language model on grapheme side...
Binary LM ready
my.arpa
my.blm


## Step 12 — Moses Training: ph→my

Source and target files are swapped compared to Step 5.




In [12]:
%%bash
mkdir -p /content/model/ph-my

echo "Starting Moses training: ph -> my"
perl /content/ubuntu-17.04/moses/scripts/training/train-model.perl \
  --root-dir /content/model/ph-my \
  --corpus /content/g2p-par/train \
  --f ph \
  --e my \
  --mgiza \
  --mgiza-cpus 2 \
  --external-bin-dir /content/ubuntu-17.04/training-tools \
  --score-options '--GoodTuring' \
  --lm 0:3:/content/model/ph-my/lm/my.blm \
  2>&1 | grep -E "^\([0-9]\)|ERROR"

echo "Training complete"

Starting Moses training: ph -> my
(1) preparing corpus @ Wed Aug  5 07:46:56 UTC 2026
(2) running giza @ Wed Aug  5 07:47:07 UTC 2026
(3) generate word alignment @ Wed Aug  5 07:47:12 UTC 2026
(4) generate lexical translation table 0-0 @ Wed Aug  5 07:47:13 UTC 2026
(5) extract phrases @ Wed Aug  5 07:47:14 UTC 2026
(6) score phrases @ Wed Aug  5 07:47:15 UTC 2026
(7) learn reordering model @ Wed Aug  5 07:47:18 UTC 2026
(8) learn generation model @ Wed Aug  5 07:47:18 UTC 2026
(9) create moses.ini @ Wed Aug  5 07:47:18 UTC 2026
Training complete


## Step 13 — Decode and Evaluate: ph→my Baseline

In [13]:
%%bash
echo "Decoding test set (ph -> my)..."
/content/ubuntu-17.04/moses/bin/moses \
  -f /content/model/ph-my/model/moses.ini \
  < /content/g2p-par/test.ph \
  > /content/model/ph-my/model/test.out \
  2>/dev/null

echo ""
echo "=== First 5 decoded outputs ==="
head -5 /content/model/ph-my/model/test.out

echo ""
echo "=== First 5 gold references ==="
head -5 /content/g2p-par/test.my

echo ""
echo "=== BLEU score: ph -> my (baseline) ==="
perl /content/ubuntu-17.04/moses/scripts/generic/multi-bleu.perl \
  /content/g2p-par/test.my \
  < /content/model/ph-my/model/test.out

Decoding test set (ph -> my)...

=== First 5 decoded outputs ===
တက် တက် ပြောင် 
ကပ် ပိ 
ရှုံ့ မဲ့ 
ညှဉ်း ပန်း 
မွမ်း မံ 

=== First 5 gold references ===
တက် တက် ပြောင်
ကပ် ပိ
ရှုံ့ မဲ့
ညှဉ်း ပန်း
မွမ်း မံ

=== BLEU score: ph -> my (baseline) ===
BLEU = 70.50, 86.8/71.9/65.1/60.8 (BP=1.000, ratio=1.000, hyp_len=8047, ref_len=8047)


## Step 14 — MERT Tuning: ph→my Improved


In [14]:
%%bash
echo "Starting MERT tuning: ph -> my"
perl /content/ubuntu-17.04/moses/scripts/training/mert-moses.pl \
  /content/g2p-par/dev.ph \
  /content/g2p-par/dev.my \
  /content/ubuntu-17.04/moses/bin/moses \
  /content/model/ph-my/model/moses.ini \
  --mertdir /content/ubuntu-17.04/moses/bin/ \
  --working-dir /content/model/ph-my/mert \
  2>&1 | grep -E "Best|BLEU|Iter|error|round"

echo "MERT finished"

Starting MERT tuning: ph -> my
exec: /content/ubuntu-17.04/moses/bin/mert -d 8  --sctype BLEU --scconfig case:true --ffile run1.features.dat --scfile run1.scores.dat --ifile run1.init.opt -n 20
Executing: /content/ubuntu-17.04/moses/bin/mert -d 8  --sctype BLEU --scconfig case:true --ffile run1.features.dat --scfile run1.scores.dat --ifile run1.init.opt -n 20 > mert.out 2> mert.log
exec: /content/ubuntu-17.04/moses/bin/mert -d 8  --sctype BLEU --scconfig case:true --ffile run1.features.dat,run2.features.dat --scfile run1.scores.dat,run2.scores.dat --ifile run2.init.opt -n 20
Executing: /content/ubuntu-17.04/moses/bin/mert -d 8  --sctype BLEU --scconfig case:true --ffile run1.features.dat,run2.features.dat --scfile run1.scores.dat,run2.scores.dat --ifile run2.init.opt -n 20 > mert.out 2> mert.log
exec: /content/ubuntu-17.04/moses/bin/mert -d 8  --sctype BLEU --scconfig case:true --ffile run1.features.dat,run2.features.dat,run3.features.dat --scfile run1.scores.dat,run2.scores.dat,run3.s

## Step 15 — Decode and Evaluate with Tuned Weights (ph→my)

In [15]:
%%bash
echo "Decoding with tuned weights (ph -> my)..."
/content/ubuntu-17.04/moses/bin/moses \
  -f /content/model/ph-my/mert/moses.ini \
  < /content/g2p-par/test.ph \
  > /content/model/ph-my/model/test.tuned.out \
  2>/dev/null

echo "=== BLEU score: ph -> my (MERT tuned) ==="
perl /content/ubuntu-17.04/moses/scripts/generic/multi-bleu.perl \
  /content/g2p-par/test.my \
  < /content/model/ph-my/model/test.tuned.out

Decoding with tuned weights (ph -> my)...
=== BLEU score: ph -> my (MERT tuned) ===
BLEU = 78.77, 88.0/79.9/75.9/72.1 (BP=1.000, ratio=1.000, hyp_len=8047, ref_len=8047)


## Step 16 — Results Summary

| Direction | Config | BLEU |
|---|---|---|
| my → ph | Baseline (default weights) | 63.35 |
| my → ph | Tuned (MERT on dev set) | *69.88* |
| ph → my | Baseline (default weights) | 70.50 |
| ph → my | Tuned (MERT on dev set) | *78.77* |



### Attribution
Data and course: Sayar Ye Kyaw Thu, AIE-F, Language Understanding Lab (LU Lab.), Myanmar.
myG2P corpus: https://github.com/ye-kyaw-thu/myG2P
